# YOLO12-Small + SPD-Conv + EMA-32 - CH-RDD2022 (Kaggle)

Varian ini memakai SPD-Conv pada downsampling P3, P4, dan P5 serta satu Efficient Multi-scale Attention (EMA-32) pada P3/8. Tidak ada ECA atau GhostConv. Notebook meng-clone branch ini agar source, parser, dan YAML yang dipakai Kaggle sama dengan repository.

Batch fisik 16 dan `nbs=64` digunakan untuk mencegah tekanan memori. `imgsz=640` dipertahankan karena SPD-Conv memerlukan ukuran gambar yang habis dibagi 32.

In [ ]:
# 1. Clone branch continuity dan install repository sebagai source training.
import json
import platform
import re
import subprocess
import sys
import zipfile
from pathlib import Path

WORKDIR = Path('/kaggle/working')
REPO_URL = 'https://github.com/danial2015/yolo-aceh-rdd2022.git'
REPO_BRANCH = 'yolo12-spd-ema32-continuity'
REPO_DIR = WORKDIR / 'yolo-aceh-rdd2022'

def log_section(title: str) -> None:
    print(f'\n{"=" * 88}\n{title}\n{"=" * 88}')

log_section('CLONE AND INSTALL MODIFIED REPOSITORY')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
REPO_METADATA = WORKDIR / 'repository_revision.txt'
REPO_METADATA.write_text(f'repository={REPO_URL}\nbranch={REPO_BRANCH}\ncommit={REPO_COMMIT}\n', encoding='utf-8')
sys.path.insert(0, str(REPO_DIR))

import torch
import ultralytics
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
log_section('ENVIRONMENT')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Commit      : {REPO_COMMIT}')
print(f'CUDA ready  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Konfigurasi dataset/hyperparameter dan verifikasi model sebelum training.
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd-2022/datasets-china-split-fix')
DATA_YAML = WORKDIR / 'ch_rdd2022.yaml'
MODEL_YAML = REPO_DIR / 'ultralytics/cfg/models/12/yolo12-spd-ema32.yaml'
CUSTOM_SOURCE_FILES = (
    REPO_DIR / 'ultralytics/nn/modules/conv.py', REPO_DIR / 'ultralytics/nn/modules/__init__.py',
    REPO_DIR / 'ultralytics/nn/tasks.py', MODEL_YAML,
)
EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS, SEED, EMA_FACTOR, EMA_INITIAL_SCALE = 0, 2, 42, 32, 1e-3
EXPERIMENT_NAME = 'yolo12s_spd_ema32_continuity_ch_rdd2022_pretrained'
RUNS_DIR = WORKDIR / 'runs'
assert IMGSZ % 32 == 0, 'SPD-Conv requires imgsz divisible by 32.'

DATA_YAML.write_text(f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair
''', encoding='utf-8')
assert DATA_ROOT.exists(), f'Dataset path tidak ditemukan: {DATA_ROOT}'
assert MODEL_YAML.exists() and all(path.exists() for path in CUSTOM_SOURCE_FILES)

from ultralytics.nn.modules import EMAAttention, GhostConv, SPDConv
from ultralytics.nn.tasks import DetectionModel
log_section('YOLO12S + SPD-CONV + EMA-32 MODEL INFO')
check_model = DetectionModel(str(MODEL_YAML), nc=5, verbose=True)
spd_layers = [layer.i for layer in check_model.model if isinstance(layer, SPDConv)]
ema_layers = [(layer.i, layer.groups) for layer in check_model.model if isinstance(layer, EMAAttention)]
ema_scales = [(layer.i, float(layer.residual_scale.detach())) for layer in check_model.model if isinstance(layer, EMAAttention)]
ghost_layers = [layer.i for layer in check_model.model if isinstance(layer, GhostConv)]
assert spd_layers == [3, 6, 8], f'SPDConv configuration is invalid: {spd_layers}'
assert ema_layers == [(5, EMA_FACTOR)], f'EMA-32 configuration is invalid: {ema_layers}'
assert all(abs(scale - EMA_INITIAL_SCALE) < 1e-9 for _, scale in ema_scales), f'EMA residual scale is invalid: {ema_scales}'
assert not ghost_layers, f'GhostConv is not part of this variant: {ghost_layers}'
check_model.eval()
with torch.inference_mode():
    model_output = check_model(torch.zeros(1, 3, IMGSZ, IMGSZ))
assert isinstance(model_output, tuple) and model_output[0].shape == (1, 9, 8400)
print(f'Parameters (5 classes): {sum(p.numel() for p in check_model.parameters()):,}')
print('SPDConv layers       :', spd_layers)
print('EMA-32 layer         :', ema_layers)
print('EMA residual scale  :', ema_scales)
check_model.info(detailed=False, verbose=True)
del check_model, model_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# 3. Transfer YOLO12s dengan kesinambungan fungsi: Conv stride-2 -> SPDConv dan EMA residual near-identity.
from ultralytics import YOLO
from ultralytics.nn.modules import Conv, EMAAttention, SPDConv

PRETRAINED_WEIGHTS = 'yolo12s.pt'
SPD_SOURCE_TARGET_PAIRS = ((3, 3), (5, 6), (7, 8))
EQUIVALENCE_IMGSZ = 320

def target_layer_index(source_index: int) -> int:
    return source_index + int(source_index >= 5)

def remap_yolo12s_weights(source_state: dict, target_state: dict) -> dict:
    transferred = {}
    pattern = re.compile(r'^model\.(\d+)(\..+)$')
    for source_key, source_tensor in source_state.items():
        match = pattern.match(source_key)
        if match is None:
            continue
        target_key = f'model.{target_layer_index(int(match.group(1)))}{match.group(2)}'
        if target_key in target_state and target_state[target_key].shape == source_tensor.shape:
            transferred[target_key] = source_tensor
    return transferred

def initialize_spd_from_pretrained_conv(source_conv: Conv, target_spd: SPDConv) -> dict:
    """Exactly reparameterize Conv(3, stride=2, padding=1) as PixelUnshuffle(2) plus Conv(3, stride=1)."""
    source_weight = source_conv.conv.weight
    target_weight = target_spd.conv.conv.weight
    compatible = (
        tuple(source_weight.shape[2:]) == (3, 3)
        and source_conv.conv.stride == (2, 2)
        and source_conv.conv.padding == (1, 1)
        and source_conv.conv.dilation == (1, 1)
        and source_conv.conv.groups == 1
        and tuple(target_weight.shape) == (source_weight.shape[0], source_weight.shape[1] * 4, 3, 3)
        and target_spd.conv.conv.stride == (1, 1)
        and target_spd.conv.conv.padding == (1, 1)
        and target_spd.conv.conv.dilation == (1, 1)
        and target_spd.conv.conv.groups == 1
    )
    if not compatible:
        raise ValueError('SPD initialization requires Conv(3x3, stride=2, padding=1, groups=1).')

    with torch.no_grad():
        target_weight.zero_()
        for source_ky in range(3):
            offset_y = source_ky - 1
            pixel_y = offset_y % 2
            target_ky = (offset_y - pixel_y) // 2 + 1
            for source_kx in range(3):
                offset_x = source_kx - 1
                pixel_x = offset_x % 2
                target_kx = (offset_x - pixel_x) // 2 + 1
                target_channels = (
                    torch.arange(source_weight.shape[1], device=target_weight.device) * 4 + pixel_y * 2 + pixel_x
                )
                target_weight[:, target_channels, target_ky, target_kx].copy_(
                    source_weight[:, :, source_ky, source_kx]
                )
        target_spd.conv.bn.load_state_dict(source_conv.bn.state_dict())

    return {
        'source_weight_shape': list(source_weight.shape),
        'target_weight_shape': list(target_weight.shape),
        'policy': 'exact Conv(s=2) -> PixelUnshuffle(2) + Conv(s=1) reparameterization',
    }

def set_ema_residual_scale(detection_model, residual_scale: float) -> list:
    initialized = []
    with torch.no_grad():
        for layer in detection_model.model:
            if isinstance(layer, EMAAttention):
                layer.residual_scale.fill_(residual_scale)
                initialized.append({'layer': int(layer.i), 'residual_scale': float(layer.residual_scale.detach())})
    if not initialized:
        raise RuntimeError('No EMAAttention layer was found in the proposed model.')
    return initialized

def verify_pretrained_continuity(source_model, target_model) -> dict:
    source_model.eval()
    target_model.eval()
    set_ema_residual_scale(target_model, 0.0)
    test_input = torch.randn(
        1, 3, EQUIVALENCE_IMGSZ, EQUIVALENCE_IMGSZ, generator=torch.Generator().manual_seed(SEED)
    )
    with torch.inference_mode():
        source_output = source_model(test_input)[0]
        target_output = target_model(test_input)[0]
    max_abs_error = float((source_output - target_output).abs().max())
    allclose = bool(torch.allclose(source_output, target_output, atol=1e-4, rtol=1e-5))
    if not allclose:
        raise AssertionError(f'Pretrained continuity check failed: max_abs_error={max_abs_error:.3e}')
    return {
        'image_size': EQUIVALENCE_IMGSZ,
        'source_output_shape': list(source_output.shape),
        'target_output_shape': list(target_output.shape),
        'max_abs_error': max_abs_error,
        'allclose': allclose,
    }

log_section('PRETRAINED CONTINUITY INITIALIZATION')
model = YOLO(str(MODEL_YAML))
source_model = YOLO(PRETRAINED_WEIGHTS).model.float()
target_state = model.model.state_dict()
transferred_state = remap_yolo12s_weights(source_model.state_dict(), target_state)
incompatible = model.model.load_state_dict(transferred_state, strict=False)

SPD_INITIALIZATION = []
for source_index, target_index in SPD_SOURCE_TARGET_PAIRS:
    source_layer = source_model.model[source_index]
    target_layer = model.model.model[target_index]
    if not isinstance(source_layer, Conv) or not isinstance(target_layer, SPDConv):
        raise TypeError(f'Expected Conv -> SPDConv at source {source_index}, target {target_index}.')
    report = initialize_spd_from_pretrained_conv(source_layer, target_layer)
    report.update({'source_layer': source_index, 'target_layer': target_index})
    SPD_INITIALIZATION.append(report)

CONTINUITY_REPORT = verify_pretrained_continuity(source_model, model.model)
EMA_INITIALIZATION = set_ema_residual_scale(model.model, EMA_INITIAL_SCALE)
PRETRAINED_REPORT = {
    'source_weights': PRETRAINED_WEIGHTS,
    'policy': 'compatible remap plus exact pretrained Conv-to-SPD reparameterization; EMA starts as a residual near-identity adapter',
    'generic_transferred_tensors': len(transferred_state),
    'target_tensors': len(target_state),
    'missing_before_spd_initialization': len(incompatible.missing_keys),
    'spd_initialization': SPD_INITIALIZATION,
    'ema_initialization': EMA_INITIALIZATION,
    'continuity_check': CONTINUITY_REPORT,
}
model.ckpt = {'model': model.model}
model.model.train()
del source_model, target_state, transferred_state
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(json.dumps(PRETRAINED_REPORT, indent=2))
print('Continuity check passed: sebelum head diganti menjadi 5 kelas, output proposed setara secara numerik dengan YOLO12s.pt.')
print('Saat fine-tuning, trainer hanya perlu menyesuaikan head deteksi dengan 5 kelas dataset.')


In [ ]:
# 4. Training dengan batch fisik 16 dan nominal batch 64.
log_section('TRAINING STARTED - YOLO12S + SPD-CONV + RESIDUAL EMA-32')
print(f'epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, nbs={NBS}, optimizer={OPTIMIZER}, lr0={LR0}, seed={SEED}')
model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, nbs=NBS, device=DEVICE, workers=WORKERS,
    project=str(RUNS_DIR), name=EXPERIMENT_NAME, exist_ok=True, pretrained=True, optimizer=OPTIMIZER,
    lr0=LR0, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, cos_lr=False, patience=PATIENCE,
    seed=SEED, plots=True, verbose=True,
)
RUN_DIR, BEST_PT, LAST_PT = Path(model.trainer.save_dir), Path(model.trainer.best), Path(model.trainer.last)
print(f'Run directory: {RUN_DIR}')
print(f'Best weights : {BEST_PT}')


In [ ]:
# 5. Evaluasi best.pt dan buat ZIP hasil.
log_section('BEST CHECKPOINT EVALUATION')
best_model = YOLO(str(BEST_PT))

def metric_summary(metrics) -> dict:
    return {'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
            'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map),
            'save_dir': str(metrics.save_dir)}

val_metrics = best_model.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                             project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_val', exist_ok=True, plots=True)
EVALUATION_REPORT = {'validation': metric_summary(val_metrics)}
test_labels = DATA_ROOT / 'test' / 'labels'
if test_labels.exists() and any(test_labels.glob('*.txt')):
    test_metrics = best_model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                                  project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_test', exist_ok=True, plots=True)
    EVALUATION_REPORT['test'] = metric_summary(test_metrics)
    TEST_OUTPUT_DIR = Path(test_metrics.save_dir)
else:
    predictions = best_model.predict(source=str(DATA_ROOT / 'test' / 'images'), imgsz=IMGSZ, device=DEVICE,
                                    conf=0.25, save=True, save_txt=True, project=str(RUNS_DIR),
                                    name=f'{EXPERIMENT_NAME}_test_predictions', exist_ok=True)
    TEST_OUTPUT_DIR = Path(predictions[0].save_dir) if predictions else RUNS_DIR
    EVALUATION_REPORT['test'] = {'status': 'labels unavailable; prediction only', 'save_dir': str(TEST_OUTPUT_DIR)}

EVALUATION_JSON = WORKDIR / f'{EXPERIMENT_NAME}_evaluation_metrics.json'
EVALUATION_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
RUN_CONFIG = WORKDIR / f'{EXPERIMENT_NAME}_config.json'
RUN_CONFIG.write_text(json.dumps({
    'dataset_root': str(DATA_ROOT), 'repository_url': REPO_URL, 'repository_branch': REPO_BRANCH,
    'repository_commit': REPO_COMMIT, 'model_yaml': str(MODEL_YAML), 'ema_factor': EMA_FACTOR,
    'spd_conv_positions': 'P3/8, P4/16, P5/32', 'pretrained_transfer': PRETRAINED_REPORT,
    'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'nbs': NBS, 'optimizer': OPTIMIZER,
    'lr0': LR0, 'momentum': MOMENTUM, 'weight_decay': WEIGHT_DECAY, 'seed': SEED,
    'best_checkpoint': str(BEST_PT), 'last_checkpoint': str(LAST_PT),
}, indent=2), encoding='utf-8')

ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'
def add_to_zip(archive: zipfile.ZipFile, path: Path) -> int:
    if not path.exists():
        return 0
    files = [path] if path.is_file() else [item for item in path.rglob('*') if item.is_file()]
    for file_path in files:
        archive.write(file_path, file_path.relative_to(WORKDIR))
    return len(files)

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    count = sum(add_to_zip(archive, Path(item)) for item in (
        RUN_DIR, Path(val_metrics.save_dir), TEST_OUTPUT_DIR, DATA_YAML, *CUSTOM_SOURCE_FILES,
        REPO_METADATA, RUN_CONFIG, EVALUATION_JSON,
    ))
print(json.dumps(EVALUATION_REPORT, indent=2))
print(f'ZIP created : {ZIP_PATH} ({count} files)')
from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))
